# Fase 3 — Transformación Silver

Bronze → Silver: limpieza, tipos, FK, reglas de negocio, `silver.rejects`.

**Orden:** pacientes → citas → eventos_clinicos → facturacion.

Prerrequisito: `01_bronze_ingesta` ejecutado.

In [ ]:
dbutils.widgets.text("repo_root", "", "Ruta Repo Databricks")
dbutils.widgets.text("batch_id", "", "Batch ID (vacío = leer de bronze.pacientes)")
dbutils.widgets.dropdown("load_mode", "full", ["full", "incremental"], "Modo carga")

In [ ]:
import sys
from uuid import uuid4

repo_root = dbutils.widgets.get("repo_root").rstrip("/")
if repo_root:
    sys.path.insert(0, f"{repo_root}/src")

from ips_analytics.config import DEFAULT_CONFIG
from ips_analytics.silver.run_silver import run_silver_pipeline

batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    batch_id = (
        spark.table(f"{DEFAULT_CONFIG.catalog}.bronze.pacientes")
        .select("_batch_id")
        .limit(1)
        .collect()[0]["_batch_id"]
    )
print(f"batch_id={batch_id}")

In [ ]:
load_mode = dbutils.widgets.get("load_mode")
result = run_silver_pipeline(
    spark, batch_id=batch_id, load_mode=load_mode, run_id=str(uuid4())
)
for e in result.entities:
    print(
        f"{e.entity}: in={e.rows_in} silver={e.rows_silver} rejected={e.rows_rejected}"
    )
print(f"TOTAL silver={result.rows_silver} rejected={result.rows_rejected}")

In [ ]:
catalog = DEFAULT_CONFIG.catalog
for t in ["pacientes", "citas", "eventos_clinicos", "facturacion"]:
    print(f"silver.{t}:", spark.table(f"{catalog}.silver.{t}").count())
try:
    print("silver.rejects:", spark.table(f"{catalog}.silver.rejects").count())
except Exception as ex:
    print("silver.rejects: N/A", ex)

## Cierre Fase 3

Ejecutar **`04_data_quality.ipynb`** para validar reglas Q-S*.

Siguiente: **Fase 4 — Gold**.